In [2]:
# base
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import warnings

# models
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor

# optimization hyperparameters
import optuna
from optuna.integration import XGBoostPruningCallback
from optuna.integration.lightgbm import LightGBMPruningCallback
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error

# log
import sys
from pathlib import Path
from src.config.log_files import auto_logger, setup_log

# path
from src.config.path import PROCESS_PATH

In [ ]:
repo_root = Path.cwd().resolve()
if repo_root.name == "eda":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

logger = setup_log(name="drone_hyperopt", filename="drone_hyperopt", stream=False)


In [ ]:
warnings.filterwarnings('ignore')

df = pd.read_parquet(PROCESS_PATH / "flight.parquet", engine="pyarrow")
df

,flight,payload,route,altitude_preset,date,time_day,duration_s,total_distance_m,wind_speed_mean,wind_speed_std,...,velocity_mag_max,acceleration_mag_mean,acceleration_mag_std,battery_voltage_mean,battery_voltage_min,max_height_agl,final_height_agl,energy_consumed_Wh,battery_needed_mAh,landing_offset_flag
0,1,0.0,R5,25,2019-04-07,10:13,200.70,549.065984,3.898058,1.952675,...,6.261616,9.842870,0.466372,22.070134,21.228519,26.257755,-0.994409,21.769975,1000.743935,False
1,2,0.0,R5,50,2019-04-07,10:23,271.20,666.615556,3.522941,2.159456,...,7.676739,9.881874,0.628406,21.527547,20.125463,52.637988,0.589477,25.366627,1205.255874,False
2,3,0.0,R5,25,2019-04-07,10:33,180.10,577.009957,4.581182,3.335733,...,7.213987,9.902090,0.545290,22.330305,19.943916,24.660462,0.106140,17.094392,789.073853,False
3,4,0.0,R5,25,2019-04-07,10:48,171.00,562.802357,4.596319,3.438072,...,9.425537,9.900368,0.559073,21.950616,20.365856,25.580572,0.416669,14.690038,687.813976,False
4,5,0.0,R2,25,2019-04-07,11:05,217.00,470.978276,3.333910,2.247522,...,4.900079,9.817243,0.341981,21.519937,18.923494,24.323036,-0.924901,19.019928,920.070980,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
204,275,500.0,R1,25,2019-10-24,9:05,149.40,518.231162,4.139878,3.885389,...,8.788151,9.850903,0.420902,22.056616,20.332052,21.959932,-3.264891,16.295811,760.480643,False
205,276,500.0,R1,25,2019-10-24,9:32,147.90,517.758034,4.392581,4.332293,...,10.553163,9.862172,0.471081,21.492353,19.788662,22.542598,1.293232,15.392088,738.620356,False
206,277,500.0,R1,25,2019-10-24,9:45,134.81,517.677109,5.524651,4.029744,...,10.579715,9.847796,0.529103,21.908016,19.352947,24.289124,-0.286362,15.531389,741.451254,False
207,278,500.0,R7,25-50-100-25,2019-10-24,10:00,186.39,545.413261,4.686967,3.826570,...,10.376503,9.829065,0.456918,22.394109,20.407175,95.041887,-0.457578,18.922540,887.198699,False


In [9]:
def log_trial_result(study, trial, value, params):
    logger.info(
        f"Hyperopt trial finished | study={study.study_name} | trial={trial.number} | value={value:.6f} | params={params}"
    )


@auto_logger(logger)
def run_hyperopt(name, obj_fn, n_trials=50):
    study = optuna.create_study(
        direction="minimize",
        study_name=f"{name}_drone_energy",
        load_if_exists=True,
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=20),
    )

    logger.info(f"Starting hyperparameter optimization for {name}")
    study.optimize(
        lambda trial: obj_fn(trial, X, y),
        n_trials=n_trials,
        callbacks=[
            lambda study, trial: log_trial_result(study, trial, trial.value, trial.params)
        ],
    )
    logger.info(
        f"Completed hyperparameter optimization for {name} | best_rmse={study.best_value:.4f} | best_params={study.best_params}"
    )
    return study

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def objective_ridge(trial, X, y):
    params = {
        "alpha": trial.suggest_float("alpha", 1e-3, 10.0, log=True),
    }
    scores = []
    for train_idx, val_idx in kf.split(X):
        model = Ridge(**params)
        model.fit(X[train_idx], y[train_idx])
        preds = model.predict(X[val_idx])
        scores.append(np.sqrt(root_mean_squared_error(y[val_idx], preds)))
    return np.mean(scores)

def objective_ada(trial, X, y):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 1.0, log=True),
        "loss": trial.suggest_categorical("loss", ["linear", "square", "exponential"]),
        "random_state": 42,
    }
    scores = []
    for train_idx, val_idx in kf.split(X):
        model = AdaBoostRegressor(**params)
        model.fit(X[train_idx], y[train_idx])
        preds = model.predict(X[val_idx])
        scores.append(np.sqrt(root_mean_squared_error(y[val_idx], preds)))
    return np.mean(scores)

def objective_rf(trial, X, y):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_depth": trial.suggest_int("max_depth", 2, 8),          # tightened: 209 rows
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 15),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "random_state": 42,
        "n_jobs": -1,
    }
    scores = []
    for train_idx, val_idx in kf.split(X):
        model = RandomForestRegressor(**params)
        model.fit(X[train_idx], y[train_idx])
        preds = model.predict(X[val_idx])
        scores.append(np.sqrt(root_mean_squared_error(y[val_idx], preds)))
    return np.mean(scores)

def objective_xgb(trial, X, y):
    params = {
        "objective": "reg:squarederror",
        "max_depth": trial.suggest_int("max_depth", 2, 6),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "random_state": 42,
        "n_jobs": -1,
    }
    scores = []
    for train_idx, val_idx in kf.split(X):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        pruning_callback = XGBoostPruningCallback(trial, "validation_0-rmse")

        model = XGBRegressor(
            **params,
            eval_metric="rmse",
            callbacks=[pruning_callback],   # moved here — constructor, not fit()
        )
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

        preds = model.predict(X_val)
        scores.append(np.sqrt(root_mean_squared_error(y_val, preds)))
    return np.mean(scores)

def objective_lgbm(trial, X, y):
    params = {
        "num_leaves": trial.suggest_int("num_leaves", 8, 40),       # tightened
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
        "min_child_samples": trial.suggest_int("min_child_samples", 3, 30),
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1,
    }
    scores = []
    for train_idx, val_idx in kf.split(X):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]
        pruning_callback = LightGBMPruningCallback(trial, "rmse")
        model = LGBMRegressor(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                  eval_metric="rmse", callbacks=[pruning_callback])
        preds = model.predict(X_val)
        scores.append(np.sqrt(root_mean_squared_error(y_val, preds)))
    return np.mean(scores)

In [ ]:
feature_cols = [
    'total_distance_m',
    'max_height_agl',
    'payload',
    'wind_speed_mean',
    'acceleration_mag_std',
    'velocity_mag_mean',
    'wind_speed_std',
    'wind_y_mean',
    'wind_x_mean',
    'acceleration_mag_mean',
] 
target = 'energy_consumed_Wh'

X = df[feature_cols].values
y = df[target].values

studies = {}
objectives = {
    "ridge": objective_ridge,
    "ada": objective_ada,
    "rf": objective_rf,
    "xgb": objective_xgb,
    "lgbm": objective_lgbm,
}

for name, obj_fn in objectives.items():
    study = run_hyperopt(name, obj_fn, n_trials=50)
    studies[name] = study
    print(f"{name} best RMSE: {study.best_value:.4f} | params: {study.best_params}")

[2026-07-15 00:56:51] [INFO] [1528722583.py:175]: Starting execution of: run_hyperopt
[I 2026-07-15 00:56:51,983] A new study created in memory with name: ridge_drone_energy
[2026-07-15 00:56:51] [INFO] [1528722583.py:38]: Starting hyperparameter optimization for ridge
[I 2026-07-15 00:56:51,993] Trial 0 finished with value: 1.6041457533567676 and parameters: {'alpha': 8.075178093115454}. Best is trial 0 with value: 1.6041457533567676.
[2026-07-15 00:56:51] [INFO] [1528722583.py:24]: Hyperopt trial finished | study=ridge_drone_energy | trial=0 | value=1.604146 | params={'alpha': 8.075178093115454}
[I 2026-07-15 00:56:52,001] Trial 1 finished with value: 1.5871961999703557 and parameters: {'alpha': 0.10139291825623414}. Best is trial 1 with value: 1.5871961999703557.
[2026-07-15 00:56:52] [INFO] [1528722583.py:24]: Hyperopt trial finished | study=ridge_drone_energy | trial=1 | value=1.587196 | params={'alpha': 0.10139291825623414}
[I 2026-07-15 00:56:52,009] Trial 2 finished with value:

ridge best RMSE: 1.5658 | params: {'alpha': 0.002866539111078059}


[I 2026-07-15 00:56:53,464] Trial 0 finished with value: 1.413639420947012 and parameters: {'n_estimators': 195, 'learning_rate': 0.8690355693936668, 'loss': 'square'}. Best is trial 0 with value: 1.413639420947012.
[2026-07-15 00:56:53] [INFO] [1528722583.py:24]: Hyperopt trial finished | study=ada_drone_energy | trial=0 | value=1.413639 | params={'n_estimators': 195, 'learning_rate': 0.8690355693936668, 'loss': 'square'}
[I 2026-07-15 00:56:55,633] Trial 1 finished with value: 1.5336316633102698 and parameters: {'n_estimators': 372, 'learning_rate': 0.010626913234845116, 'loss': 'square'}. Best is trial 0 with value: 1.413639420947012.
[2026-07-15 00:56:55] [INFO] [1528722583.py:24]: Hyperopt trial finished | study=ada_drone_energy | trial=1 | value=1.533632 | params={'n_estimators': 372, 'learning_rate': 0.010626913234845116, 'loss': 'square'}
[I 2026-07-15 00:56:58,159] Trial 2 finished with value: 1.4117650433825064 and parameters: {'n_estimators': 493, 'learning_rate': 0.06791236

ada best RMSE: 1.3636 | params: {'n_estimators': 413, 'learning_rate': 0.9789731440027141, 'loss': 'exponential'}


[I 2026-07-15 00:58:16,288] Trial 0 finished with value: 1.8262479987379017 and parameters: {'n_estimators': 215, 'max_depth': 3, 'min_samples_split': 7, 'min_samples_leaf': 9, 'max_features': 'log2'}. Best is trial 0 with value: 1.8262479987379017.
[2026-07-15 00:58:16] [INFO] [1528722583.py:24]: Hyperopt trial finished | study=rf_drone_energy | trial=0 | value=1.826248 | params={'n_estimators': 215, 'max_depth': 3, 'min_samples_split': 7, 'min_samples_leaf': 9, 'max_features': 'log2'}
[I 2026-07-15 00:58:18,197] Trial 1 finished with value: 1.3857633922759642 and parameters: {'n_estimators': 432, 'max_depth': 8, 'min_samples_split': 4, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 1 with value: 1.3857633922759642.
[2026-07-15 00:58:18] [INFO] [1528722583.py:24]: Hyperopt trial finished | study=rf_drone_energy | trial=1 | value=1.385763 | params={'n_estimators': 432, 'max_depth': 8, 'min_samples_split': 4, 'min_samples_leaf': 3, 'max_features': 'sqrt'}
[I 2026-07-15 00

rf best RMSE: 1.3660 | params: {'n_estimators': 410, 'max_depth': 8, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None}


[I 2026-07-15 00:59:38,540] Trial 0 finished with value: 1.2889090395380365 and parameters: {'max_depth': 4, 'learning_rate': 0.025430591411263765, 'subsample': 0.9748185474801352, 'colsample_bytree': 0.879574604296053, 'reg_alpha': 1.370421326296934e-06, 'reg_lambda': 1.2605982573915822e-05, 'min_child_weight': 1, 'n_estimators': 179}. Best is trial 0 with value: 1.2889090395380365.
[2026-07-15 00:59:38] [INFO] [1528722583.py:24]: Hyperopt trial finished | study=xgb_drone_energy | trial=0 | value=1.288909 | params={'max_depth': 4, 'learning_rate': 0.025430591411263765, 'subsample': 0.9748185474801352, 'colsample_bytree': 0.879574604296053, 'reg_alpha': 1.370421326296934e-06, 'reg_lambda': 1.2605982573915822e-05, 'min_child_weight': 1, 'n_estimators': 179}
[I 2026-07-15 00:59:39,245] Trial 1 finished with value: 1.2697759505882213 and parameters: {'max_depth': 3, 'learning_rate': 0.018981905873656075, 'subsample': 0.892294374006031, 'colsample_bytree': 0.8541027752812367, 'reg_alpha': 

xgb best RMSE: 1.2674 | params: {'max_depth': 3, 'learning_rate': 0.07161183556408778, 'subsample': 0.9699580300221706, 'colsample_bytree': 0.8410040190795652, 'reg_alpha': 8.486478832504852e-06, 'reg_lambda': 9.888875194983115e-07, 'min_child_weight': 2, 'n_estimators': 385}


[I 2026-07-15 00:59:57,621] Trial 0 finished with value: 1.5812181281929956 and parameters: {'num_leaves': 32, 'max_depth': 8, 'learning_rate': 0.035275248152508966, 'feature_fraction': 0.6312855424827706, 'bagging_fraction': 0.906188281650715, 'min_child_samples': 22, 'n_estimators': 279}. Best is trial 0 with value: 1.5812181281929956.
[2026-07-15 00:59:57] [INFO] [1528722583.py:24]: Hyperopt trial finished | study=lgbm_drone_energy | trial=0 | value=1.581218 | params={'num_leaves': 32, 'max_depth': 8, 'learning_rate': 0.035275248152508966, 'feature_fraction': 0.6312855424827706, 'bagging_fraction': 0.906188281650715, 'min_child_samples': 22, 'n_estimators': 279}
[I 2026-07-15 00:59:58,038] Trial 1 finished with value: 1.5006396730595157 and parameters: {'num_leaves': 40, 'max_depth': 3, 'learning_rate': 0.20943352111571187, 'feature_fraction': 0.81331644493622, 'bagging_fraction': 0.8362608352099004, 'min_child_samples': 25, 'n_estimators': 349}. Best is trial 1 with value: 1.500639

lgbm best RMSE: 1.1539 | params: {'num_leaves': 8, 'max_depth': 2, 'learning_rate': 0.21685220605980327, 'feature_fraction': 0.8029048955724699, 'bagging_fraction': 0.6529537513693758, 'min_child_samples': 3, 'n_estimators': 384}


In [7]:
Ridge(
    alpha=0.002866539111078059,
    random_state=42,
)

AdaBoostRegressor(
    n_estimators=413,
    learning_rate=0.9789731440027141,
    loss='exponential',
    random_state=42,
)

RandomForestRegressor(
    n_estimators=410,
    max_depth=8,
    min_samples_split=5,
    min_samples_leaf=3,
    n_jobs=-1,
    random_state=42,
)

XGBRegressor(
    max_depth=3,
    learning_rate=0.07161183556408778,
    subsample=0.9699580300221706,
    colsample_bytree=0.8410040190795652,
    reg_alpha=0.8410040190795652,
    reg_lambda=9.888875194983115e-07,
    min_child_weight=2,
    n_estimators=385,
    n_jobs=-1,
    random_state=42,
)

LGBMRegressor(
    num_leaves=8,
    max_depth=2,
    learning_rate=0.21685220605980327,
    feature_fraction=0.8029048955724699,
    bagginh_fraction=0.6529537513693758,
    min_child_samples=3,
    n_estimators=384,
    n_jobs=-1,
    random_state=42,
    verbose=-1,
)

,num_leaves,8
,max_depth,2
,learning_rate,0.21685220605980327
,n_estimators,384
,min_child_samples,3
,random_state,42
,n_jobs,-1
,feature_fraction,0.8029048955724699
,bagginh_fraction,0.6529537513693758
,boosting_type,'gbdt'
,subsample_for_bin,200000


In [8]:
del df
gc.collect()
%whos

Variable                         Type         Data/Info
-------------------------------------------------------
AdaBoostRegressor                ABCMeta      <class 'sklearn.ensemble.<...>sting.AdaBoostRegressor'>
KFold                            ABCMeta      <class 'sklearn.model_selection._split.KFold'>
LGBMRegressor                    type         <class 'lightgbm.sklearn.LGBMRegressor'>
LightGBMPruningCallback          type         <class 'optuna_integratio<...>LightGBMPruningCallback'>
Path                             type         <class 'pathlib.Path'>
RandomForestRegressor            ABCMeta      <class 'sklearn.ensemble.<...>t.RandomForestRegressor'>
Ridge                            ABCMeta      <class 'sklearn.linear_model._ridge.Ridge'>
X                                ndarray      209x10: 2090 elems, type `float64`, 16720 bytes
XGBRegressor                     type         <class 'xgboost.sklearn.XGBRegressor'>
XGBoostPruningCallback           ABCMeta      <class 'optuna_int